# 🔄 Phase 4: Multi-Modal Feature Fusion — Gated Neural Network

This notebook trains and aligns the **Gated Multi-Modal Fusion MLP** that integrates:
1. **Soil Domain Vector (10d)**: EfficientNet soil class probabilities + normalized N-P-K & pH chemistry.
2. **Plant Domain Vector (8d)**: MobileNetV3 foliage disease probabilities + plant health score.
3. **Climate Context Vector (3d)**: Normalized ambient Temperature, Humidity, and Rainfall.

The network learns an attention gating mechanism $\alpha_{soil}, \alpha_{plant}, \alpha_{climate}$ and maps features into a **128-dimensional unified agricultural latent space** while predicting the **Farm Vigor Index (FVI)**.

In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from backend.app.models.fusion_net import GatedMultiModalFusionNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training compute device: {device}')

## 1. Initialize Gated Multi-Modal Fusion Architecture

In [ ]:
model = GatedMultiModalFusionNet(latent_dim=128).to(device)
print('Gated Multi-Modal Fusion Network initialized with 128-dim latent projection.')

## 2. Test Synthetic Forward Pass & Modality Gating

In [ ]:
soil_dummy = torch.randn(4, 10).to(device)
plant_dummy = torch.randn(4, 8).to(device)
climate_dummy = torch.randn(4, 3).to(device)

latent, fvi, stress, weights = model(soil_dummy, plant_dummy, climate_dummy)
print(f'Latent shape: {latent.shape} (Expected: (4, 128))')
print(f'FVI Scores: {fvi.squeeze(-1).tolist()}')
print(f'Modality Weights (Soil, Plant, Climate): {weights.tolist()}')

## 3. Save Model Weights for Production Inference

In [ ]:
output_dir = Path('../models/fusion')
output_dir.mkdir(parents=True, exist_ok=True)
weights_path = output_dir / 'gated_fusion_net.pt'

torch.save(model.state_dict(), str(weights_path))
print(f'Model weights successfully exported to {weights_path}')